<img src='https://www.icos-cp.eu/sites/default/files/2017-11/ICOS_CP_logo.png' width=400 align=right>

# ICOS Carbon Portal Python Libraries

This example uses a foundational library called `icoscp_core` which can be used to access time-series ICOS data that are *previewable* in the ICOS Data Portal. "Previewable" means that it is possible to visualize the data variables in the preview plot. The library can also be used to access (meta-)data from [ICOS Cities](https://citydata.icos-cp.eu/portal/) and [SITES](https://data.fieldsites.se/portal/) data repositories.

General information on all ICOS Carbon Portal Python libraries can be found on our [help pages](https://icos-carbon-portal.github.io/pylib/). Documentation of the `icoscp_core` library, including information on running it locally, can also be found on [PyPI.org](https://pypi.org/project/icoscp_core/).

# Example: access and work with atmospheric Level 2 data

> **All the choices are made through dropdowns.** Station, data type, data object, and the columns to
> plot are selected interactively; nothing has to be copied by hand from one cell to another.

**Folder management: to avoid accumulating files across repeated runs, this notebook empties its `outputs/` and `exports/` folders at the start of each run. Use the export cell at the end to keep anything you want before running it again.**

> **Clearing the outputs.** Use *Edit → Clear Outputs of All Cells* in two situations:
>
> * **before running the notebook again**, so that the interactive widgets are rebuilt from scratch.
>   Widget state belongs to the kernel that created it, so leftovers from a previous run can make
>   controls stop responding or disappear;
> * **before saving**, so that the file stays small and does not carry stale widget state. Plotly
>   embeds a large JavaScript bundle into saved outputs, and reopening a notebook that stores widget
>   state shows "Error displaying widget: model not found".
>
> If controls are still unresponsive after clearing, restart the kernel (*Kernel → Restart Kernel*)
> and, should that not be enough, reload the browser page: the widgets live partly in the page
> itself, and a reload is sometimes the only way to clear what is left there.

## 0. Configuration

Only fallbacks and display options live here — the actual selections are made in the dropdowns below.

* `DEFAULT_COUNTRY`, `DEFAULT_STATION_CODE`, `DEFAULT_DATA_TYPE_KEYWORD` — used if you skip a selector
* `MAP_BACKEND` — `"plotly"` for the interactive station map, `"matplotlib"` if it stays blank
* `MAX_OBJECTS_TO_COMBINE` — safety limit for the section that fetches every data object

In [ ]:
# The kernel keeps a handle on the directory it was started in. If that directory is replaced
# while the kernel is running - the shared workspace being republished, for instance - every
# relative path fails with "Stale file handle" or FileNotFoundError. Recover instead of leaving
# the user with an error that has nothing to do with what they were doing.
import os

try:
    os.getcwd()
except OSError:
    _nb = os.environ.get("JPY_SESSION_NAME", "")
    _target = os.path.dirname(_nb) if _nb else os.path.expanduser("~")
    if not os.path.isdir(_target):
        _target = os.path.expanduser("~")
    os.chdir(_target)
    print(f"The working directory had become stale and was reset to {_target}.")
    print("If anything still looks wrong, restart the kernel (Kernel -> Restart Kernel).")

# Identity of this notebook. The file itself carries no version number - only one version is
# published at a time - so the version is printed here instead, which is also what tells you
# whether the kernel is running the file you think it is.
NOTEBOOK_NAME = "ICOS atmospheric Level 2"
NOTEBOOK_VERSION = "2.2"
NOTEBOOK_DATE = "2026-09-15"

print(f"{NOTEBOOK_NAME} - version {NOTEBOOK_VERSION} ({NOTEBOOK_DATE})")

DEFAULT_COUNTRY = "IT"
DEFAULT_STATION_CODE = "CMN"
# Matched as a substring against the data-type label, so it has to be specific enough:
# "ATC CO" would also match "ICOS ATC CO2 Release".
DEFAULT_DATA_TYPE_KEYWORD = "ATC CO Release"

# Authentication token (optional). Leave empty to use Carbon Portal credentials instead.
# See section 2: a token is the way in for accounts that sign in through ORCID or an
# institutional provider and therefore have no Carbon Portal password.
#   ICOS_COOKIE_TOKEN = "cpauthToken=<token from My Account>"
# It expires in under 28 hours, and it is written in the notebook source: never leave it in
# a copy that goes back to the shared workspace.
ICOS_COOKIE_TOKEN = ""

MAP_BACKEND = "plotly"
MAX_OBJECTS_TO_COMBINE = 100

# Column conventions of the ATC Level 2 products. Neither the time column nor the quality-flag
# column is the same across products - flask data use "SamplingStart", the hourly time series
# use "TIMESTAMP" - so both are resolved against the data once it is loaded. The names below
# are only the candidates and the fallback.
TIME_COLUMN_CANDIDATES = ("SamplingStart", "TIMESTAMP", "TimeStamp", "time", "Date")
TIME_COLUMN = TIME_COLUMN_CANDIDATES[0]
DEFAULT_QUALITY_FLAG = "Flag"
VALUE_ACCEPT_QUALITY = "O"

NOTEBOOK_SLUG = "icos_L2"


## 1. Import libraries

In [ ]:
import os
import importlib.util
import math
import subprocess
import sys

# Install only what is missing. Each VRE user has their own container, so a notebook that runs
# for a colleague can still fail here with ModuleNotFoundError.
REQUIRED = {
    "icoscp_core": "icoscp",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "plotly": "plotly",
    "ipywidgets": "ipywidgets",
    # Hidden dependency: plotly needs nbformat to render figures but does not declare it.
    "nbformat": "nbformat",
}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"Installing: {', '.join(missing)}")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing],
                       capture_output=True, text=True,
                            cwd=os.path.expanduser("~"))
    print("Done." if r.returncode == 0 else r.stderr[-1500:])
else:
    print("All required packages are already installed.")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode
import ipywidgets as widgets
from IPython.display import display

from icoscp_core.icos import data, meta, ATMO_STATION

# Loads the Plotly JavaScript into the page. Without it, figures shown inside an ipywidgets
# Output (the station map) render as an empty area. connected=True pulls it from the CDN
# instead of embedding several MB into the saved notebook.
if MAP_BACKEND == "plotly":
    init_notebook_mode(connected=True)


# --- Outputs of this run -------------------------------------------------------------
# Figures are shown inline and kept in memory; nothing is written to disk while you work.
# The export cell at the end of the notebook writes out what you decide to keep, into the
# folder below, and packs it into a single zip you can download.
#
# The folder is emptied at the start of each run, so that repeated runs do not accumulate
# files. Set CLEAN_OUTPUTS to False to keep what earlier runs produced.
import shutil
from pathlib import Path

OUTPUT_DIR = Path("outputs")
CLEAN_OUTPUTS = True

if CLEAN_OUTPUTS and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
        print("Note: some files could not be removed (still open elsewhere); continuing anyway.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DRAWN_FIGURES = {}


def register_figure(fig, stem):
    """Remember a figure under a name, for the export cell at the end."""
    DRAWN_FIGURES[stem] = fig

# Archives produced by the export cell at the end go here. The folder is emptied together with
# the rest, so the zips of previous runs do not pile up next to the notebook.
EXPORT_DIR = Path("exports")
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR, ignore_errors=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Authentication

Accessing ICOS data through the `icoscp` library requires authentication, which is how the Carbon
Portal records that you have accepted the data licence. You need an account at
[cpauth.icos-cp.eu](https://cpauth.icos-cp.eu/) and you must accept the ICOS Data Licence in your
user profile. There are two ways to authenticate.

### Option A — Carbon Portal credentials (default, persistent)

The cell below asks for your Carbon Portal email and password and stores the configuration in your
home directory, so you are only asked once. This requires an account whose password the Carbon
Portal itself can verify: if you normally sign in through a federated provider (ORCID, institutional
account) and have never set a password, either set one in your profile, or use option B.

### Option B — authentication token (works with any login method, temporary)

A token works regardless of how you sign in, which makes it the way in for federated accounts. Log
in at [cpauth.icos-cp.eu](https://cpauth.icos-cp.eu/) with whichever mechanism you use, open **My
Account**, copy the token, and set it in the Configuration cell:

```python
ICOS_COOKIE_TOKEN = "cpauthToken=<token from My Account>"
```

Two caveats. The token expires after **less than 28 hours**, so it suits a working session rather
than a permanent setup. And it is written in the notebook source: **do not leave it in a notebook
you copy back into the shared workspace**, where every VRE user could read and reuse it until it
expires. Keep it in your own copy in your home directory and clear the variable before sharing.

Either way, this cell is safe to re-run: if valid credentials are already available it says so and
asks for nothing.

In [ ]:
from icoscp_core.icos import auth, bootstrap


def _authenticated() -> bool:
    """auth.get_token() raises when no usable credentials are stored, so it doubles as a probe."""
    try:
        auth.get_token()
        return True
    except Exception:
        return False


if ICOS_COOKIE_TOKEN.strip():
    # Option B. bootstrap.fromCookieToken returns *new* metadata and data clients: they have to
    # replace the ones imported in section 1, otherwise the rest of the notebook would keep
    # using the unauthenticated connection and fail later, far from the cause.
    try:
        meta, data = bootstrap.fromCookieToken(ICOS_COOKIE_TOKEN.strip())
        meta.list_stations(ATMO_STATION)      # cheap call, fails fast if the token is invalid
        print("Authenticated with the token from the Configuration cell.")
        print("Remember: it expires in under 28 hours, and must not be left in a shared copy "
              "of this notebook.")
    except Exception as exc:  # noqa: BLE001 - an expired or mistyped token must be readable
        raise RuntimeError(
            f"The token in ICOS_COOKIE_TOKEN was not accepted ({type(exc).__name__}: {exc}).\n"
            "Tokens expire in under 28 hours - fetch a fresh one from the 'My Account' page at "
            "https://cpauth.icos-cp.eu/, or clear ICOS_COOKIE_TOKEN to use option A instead."
        ) from exc

elif _authenticated():
    print("Already authenticated with stored credentials - nothing to do.")

else:
    # Option A: prompts once, then stores the configuration in the home directory
    try:
        auth.init_config_file()
    except Exception as exc:  # noqa: BLE001 - authentication failures should be readable
        print(f"Authentication setup failed ({type(exc).__name__}: {exc})")

    if _authenticated():
        print("\nAuthentication configured - move on to the next section.")
    else:
        print("\nStill not authenticated. Either set a password for your Carbon Portal account "
              "at https://cpauth.icos-cp.eu/ and re-run this cell, or use a token (option B "
              "above) by setting ICOS_COOKIE_TOKEN in the Configuration cell.")

## 3. Select the station

Pick a **country**, then a **station**, and press **Confirm station**. The map shows the stations of
the selected country with the chosen one in red.

⚠️ Picking a value is not enough — press the button, or the notebook falls back to
`DEFAULT_STATION_CODE` and says so. Re-running this cell keeps an already-confirmed station.

In [ ]:
stations = meta.list_stations(ATMO_STATION)
print(f"{len(stations)} ATMO stations available.")
stations_by_code = {s.id: s for s in stations}

def _country_options():
    codes = sorted({s.country_code for s in stations if s.country_code})
    return [("All countries", "ALL")] + [(c, c) for c in codes]

def _stations_for(country):
    sel = [s for s in stations if country == "ALL" or s.country_code == country]
    return sorted((f"{s.id} - {s.name} ({s.country_code})", s.id) for s in sel)

if "SELECTED_STATION" not in globals():
    SELECTED_STATION = None
_initial = SELECTED_STATION if SELECTED_STATION in stations_by_code else DEFAULT_STATION_CODE
if _initial not in stations_by_code:
    _initial = stations[0].id
_initial_country = stations_by_code[_initial].country_code or "ALL"

country_dd = widgets.Dropdown(options=_country_options(), value=_initial_country,
                              description="Country:", style={"description_width": "initial"},
                              layout=widgets.Layout(width="320px"))
station_dd = widgets.Dropdown(options=_stations_for(_initial_country), value=_initial,
                              description="Station:", style={"description_width": "initial"},
                              layout=widgets.Layout(width="620px"))
confirm_btn = widgets.Button(description="Confirm station", button_style="success", icon="check")
confirm_out = widgets.Output()
map_out = widgets.Output()
_updating = {"busy": False}

GEO = dict(resolution=50, showcountries=True, countrycolor="#888", showland=True,
           landcolor="#f2efe9", showocean=True, oceancolor="#dbe9f2", showlakes=False)

def _draw_map(*_):
    with map_out:
        map_out.clear_output(wait=True)
        shown = [stations_by_code[c] for _, c in station_dd.options]
        located = [s for s in shown if s.lat is not None and s.lon is not None]
        if not located:
            print("No coordinates available for these stations.")
            return
        current = station_dd.value
        others = [s for s in located if s.id != current]
        selected = [s for s in located if s.id == current]

        if MAP_BACKEND == "plotly":
            fig = go.Figure()
            if others:
                fig.add_trace(go.Scattergeo(
                    lon=[s.lon for s in others], lat=[s.lat for s in others],
                    text=[f"{s.id} - {s.name}" for s in others], hoverinfo="text",
                    mode="markers",
                    marker=dict(size=8, color="#3B4872", line=dict(width=1, color="white"))))
            if selected:
                fig.add_trace(go.Scattergeo(
                    lon=[s.lon for s in selected], lat=[s.lat for s in selected],
                    text=[f"{s.id} - {s.name}" for s in selected], hoverinfo="text",
                    mode="markers",
                    marker=dict(size=16, color="crimson", symbol="star",
                                line=dict(width=1.5, color="white"))))
            fig.update_geos(fitbounds="locations", **GEO)
            fig.update_layout(height=420, margin=dict(l=0, r=0, t=36, b=0), showlegend=False,
                              title=dict(text=f"{len(located)} station(s) - selected: {current}",
                                         font=dict(size=13)))
            fig.show()
        else:
            lons = [s.lon for s in located]; lats = [s.lat for s in located]
            fig, ax = plt.subplots(figsize=(7, 4.5))
            if others:
                ax.scatter([s.lon for s in others], [s.lat for s in others], s=45,
                           c="#3B4872", edgecolors="white", zorder=3)
            if selected:
                ax.scatter([s.lon for s in selected], [s.lat for s in selected], s=160,
                           c="crimson", marker="*", edgecolors="white", zorder=4)
            for s in (located if len(located) <= 25 else selected):
                ax.annotate(s.id, (s.lon, s.lat), textcoords="offset points",
                            xytext=(6, 4), fontsize=8)
            pad = max(1.5, 0.15 * max(max(lons) - min(lons), max(lats) - min(lats)))
            ax.set_xlim(min(lons) - pad, max(lons) + pad)
            ax.set_ylim(min(lats) - pad, max(lats) + pad)
            ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.grid(True, linestyle=":")
            ax.set_title(f"{len(located)} station(s) - selected: {current}", fontsize=11)
            ax.set_aspect(1.0 / max(0.2, math.cos(math.radians(sum(lats) / len(lats)))),
                          adjustable="box")
            plt.tight_layout(); plt.show()

def _on_country(change):
    opts = _stations_for(change["new"])
    _updating["busy"] = True
    try:
        station_dd.options = opts
        if opts:
            station_dd.value = opts[0][1]
    finally:
        _updating["busy"] = False
    _draw_map()

def _on_station(change):
    if not _updating["busy"]:
        _draw_map()

def _on_confirm(_):
    global SELECTED_STATION
    SELECTED_STATION = station_dd.value
    s = stations_by_code[SELECTED_STATION]
    with confirm_out:
        confirm_out.clear_output()
        print(f"Confirmed station: {s.id} - {s.name} ({s.country_code})")

country_dd.observe(_on_country, names="value")
station_dd.observe(_on_station, names="value")
confirm_btn.on_click(_on_confirm)
display(widgets.VBox([country_dd, station_dd, map_out, confirm_btn, confirm_out]))
_draw_map()

with confirm_out:
    if SELECTED_STATION:
        print(f"Confirmed station: {SELECTED_STATION} (already confirmed - you can move on)")
    else:
        print(f"Not confirmed yet - will default to {DEFAULT_STATION_CODE}.")

### Apply the selection

The station metadata and both spellings of its URI are derived here.

In [ ]:
SELECTED_STATION = globals().get("SELECTED_STATION")
if SELECTED_STATION not in stations_by_code:
    SELECTED_STATION = DEFAULT_STATION_CODE
    print(f"No station confirmed - defaulting to {SELECTED_STATION}.")

station_obj = stations_by_code[SELECTED_STATION]
STATION_CODE = station_obj.id

# The station's own URI is used rather than a hand-written string. The metadata service wants
# https for station metadata but http in data-object queries, so both are derived from it here
# instead of being typed twice in different cells.
_uri = station_obj.uri
STATION_URI_HTTPS = _uri.replace("http://", "https://", 1)
STATION_URI_HTTP = _uri.replace("https://", "http://", 1)

station_meta = meta.get_station_meta(STATION_URI_HTTPS)
station_name = station_meta.org.name

print(f"Station: {STATION_CODE} - {station_name} ({station_obj.country_code})")
print("Got ICOS label on:", station_meta.specificInfo.labelingDate)

print("\nKnown staff, possibly former:")
display(pd.DataFrame([
    {"first_name": m.person.firstName, "last_name": m.person.lastName,
     "role": m.role.role.label, "start": m.role.start, "end": m.role.end}
    for m in station_meta.staff
]))

## 4. Select the data type

Data types combine a number of metadata elements, so that data objects are associated with a type
instead of being linked to numerous separate fields. The dropdown lists the previewable
**Level 2** data types of the atmospheric domain. See more on
[data levels](https://www.icos-cp.eu/data-services/data-collection/data-levels-quality).

In [ ]:
all_data_types = meta.list_datatypes()
selected_datatypes = [
    dt for dt in all_data_types
    if dt.has_data_access and dt.data_level == 2
    and dt.theme.uri == "http://meta.icos-cp.eu/resources/themes/atmosphere"
]
print(f"{len(selected_datatypes)} previewable Level 2 atmospheric data type(s).")
datatypes_by_uri = {dt.uri: dt for dt in selected_datatypes}

_dt_options = sorted((dt.label, dt.uri) for dt in selected_datatypes)

if "SELECTED_DATA_TYPE" not in globals():
    SELECTED_DATA_TYPE = None
_initial_dt = SELECTED_DATA_TYPE if SELECTED_DATA_TYPE in datatypes_by_uri else None
if _initial_dt is None:
    _match = [u for lbl, u in _dt_options if DEFAULT_DATA_TYPE_KEYWORD.lower() in lbl.lower()]
    _initial_dt = _match[0] if _match else _dt_options[0][1]

datatype_dd = widgets.Dropdown(options=_dt_options, value=_initial_dt, description="Data type:",
                               style={"description_width": "initial"},
                               layout=widgets.Layout(width="760px"))
confirm_dt_btn = widgets.Button(description="Confirm data type", button_style="success", icon="check")
confirm_dt_out = widgets.Output()

def _on_confirm_dt(_):
    global SELECTED_DATA_TYPE
    SELECTED_DATA_TYPE = datatype_dd.value
    with confirm_dt_out:
        confirm_dt_out.clear_output()
        print(f"Confirmed: {datatypes_by_uri[SELECTED_DATA_TYPE].label}")

confirm_dt_btn.on_click(_on_confirm_dt)
display(widgets.VBox([datatype_dd, confirm_dt_btn, confirm_dt_out]))

with confirm_dt_out:
    if SELECTED_DATA_TYPE:
        print("Already confirmed - you can move on.")
    else:
        print(f"Not confirmed yet - will default to the first type matching "
              f"'{DEFAULT_DATA_TYPE_KEYWORD}'.")

### Apply the selection and list the matching data objects

In [ ]:
SELECTED_DATA_TYPE = globals().get("SELECTED_DATA_TYPE")
if SELECTED_DATA_TYPE not in datatypes_by_uri:
    SELECTED_DATA_TYPE = _initial_dt
    print(f"No data type confirmed - defaulting to {datatypes_by_uri[SELECTED_DATA_TYPE].label}.")

DATA_TYPE = SELECTED_DATA_TYPE
print(f"Data type: {datatypes_by_uri[DATA_TYPE].label}")

station_data_objects = meta.list_data_objects(datatype=DATA_TYPE, station=STATION_URI_HTTP)
print(f"{len(station_data_objects)} data object(s) for {STATION_CODE}.")

if not station_data_objects:
    # Not every station provides every product: report where the data actually is.
    print(f"\n{STATION_CODE} has no data of this type.")
    try:
        elsewhere = meta.list_data_objects(datatype=DATA_TYPE)
    except Exception as exc:  # noqa: BLE001
        elsewhere = []
        print(f"  (could not check other stations: {type(exc).__name__}: {exc})")
    codes = sorted({o.station_uri.split("_")[-1] for o in elsewhere if o.station_uri})
    if codes:
        named = [f"{c} ({stations_by_code[c].name})" if c in stations_by_code else c for c in codes]
        print(f"  available at {len(codes)} station(s): {', '.join(named)}")
        hint = f"available at {', '.join(codes)}"
    else:
        print("  no data objects at all for this data type - it appears to be empty.")
        hint = "this data type is empty across the archive"
    raise RuntimeError(f"No data objects for {STATION_CODE} / "
                       f"{datatypes_by_uri[DATA_TYPE].label}. Reason: {hint}.")

## 5. Choose a data object

The dropdown is built from the objects listed above, showing the sampling height and the filename.
The height comes from the object metadata (`sampling_height`), not from parsing the filename, so it
stays correct whatever naming convention a product uses.

In [ ]:
def _obj_label(o):
    h = f"{o.sampling_height} m" if o.sampling_height is not None else "height n/a"
    return f"{h}  |  {o.filename}"

_obj_options = sorted((_obj_label(o), o.filename) for o in station_data_objects)
objects_by_filename = {o.filename: o for o in station_data_objects}

if "SELECTED_FILE" not in globals():
    SELECTED_FILE = None
_initial_file = SELECTED_FILE if SELECTED_FILE in objects_by_filename else _obj_options[0][1]

file_dd = widgets.Dropdown(options=_obj_options, value=_initial_file, description="Data object:",
                           style={"description_width": "initial"},
                           layout=widgets.Layout(width="860px"))
confirm_file_btn = widgets.Button(description="Confirm data object", button_style="success",
                                  icon="check")
confirm_file_out = widgets.Output()

def _on_confirm_file(_):
    global SELECTED_FILE
    SELECTED_FILE = file_dd.value
    with confirm_file_out:
        confirm_file_out.clear_output()
        print(f"Confirmed: {SELECTED_FILE}")

confirm_file_btn.on_click(_on_confirm_file)
display(widgets.VBox([file_dd, confirm_file_btn, confirm_file_out]))

with confirm_file_out:
    if SELECTED_FILE in objects_by_filename:
        print(f"Confirmed: {SELECTED_FILE} (already confirmed - you can move on)")
    else:
        print("Not confirmed yet - will default to the first object in the list.")

### Apply the selection and load the data

In [ ]:
SELECTED_FILE = globals().get("SELECTED_FILE")
if SELECTED_FILE not in objects_by_filename:
    SELECTED_FILE = _obj_options[0][1]
    print(f"No data object confirmed - defaulting to {SELECTED_FILE}")

selected_data_object = objects_by_filename[SELECTED_FILE]
print(f"File: {selected_data_object.filename}")

# Full metadata is required to fetch the data; this is a relatively expensive call (~100 ms)
dobj_meta = meta.get_dobj_meta(selected_data_object)
df = pd.DataFrame(data.get_columns_as_arrays(dobj_meta))


def resolve_time_column(frame):
    """Find the time column of this product.

    Different ATC Level 2 products name it differently, so it is taken from the data rather
    than assumed: a fixed name works for flask data and fails with a bare KeyError on the
    hourly series.
    """
    for name in TIME_COLUMN_CANDIDATES:
        if name in frame.columns:
            return name
    for name in frame.columns:                       # anything already parsed as a date
        if pd.api.types.is_datetime64_any_dtype(frame[name]):
            return name
    raise RuntimeError(
        f"No time column found among {list(frame.columns)}. Add its name to "
        f"TIME_COLUMN_CANDIDATES in the Configuration cell."
    )


TIME_COLUMN = resolve_time_column(df)
print(f"Time column: {TIME_COLUMN}")

display(df)

## 6. Choose the column to plot

The dropdown lists the numeric columns of the loaded object, with the variable name and unit taken
from the metadata, so you do not have to guess what a column such as `co2` or `14C` contains.

In [ ]:
ts_meta = dobj_meta.specificInfo
column_value_types = {col.label: col.valueType for col in ts_meta.columns}

def _col_label(col):
    vt = column_value_types.get(col)
    if vt is None:
        return col
    unit = f" [{vt.unit}]" if getattr(vt, "unit", None) else ""
    return f"{col} - {vt.self.label}{unit}"

plottable = [c for c in df.columns
             if c != TIME_COLUMN and not c.endswith("Flag")
             and pd.api.types.is_numeric_dtype(df[c])]
if not plottable:
    raise RuntimeError(f"No numeric column to plot. Columns: {list(df.columns)}")

# Ancillary columns come first alphabetically (NbPoints, Stdev), so the plain first entry
# would open the notebook on the number of points rather than on the measurement itself.
ANCILLARY = ("NbPoints", "Stdev", "Flag")
_measurements = [c for c in plottable if c not in ANCILLARY]

if "SELECTED_COLUMN" not in globals():
    SELECTED_COLUMN = None
_initial_col = (SELECTED_COLUMN if SELECTED_COLUMN in plottable
                else (_measurements[0] if _measurements else plottable[0]))

column_dd = widgets.Dropdown(options=[(_col_label(c), c) for c in plottable], value=_initial_col,
                             description="Column:", style={"description_width": "initial"},
                             layout=widgets.Layout(width="620px"))
confirm_col_btn = widgets.Button(description="Confirm column", button_style="success", icon="check")
confirm_col_out = widgets.Output()

def _on_confirm_col(_):
    global SELECTED_COLUMN
    SELECTED_COLUMN = column_dd.value
    with confirm_col_out:
        confirm_col_out.clear_output()
        print(f"Confirmed column: {_col_label(SELECTED_COLUMN)}")

confirm_col_btn.on_click(_on_confirm_col)
display(widgets.VBox([column_dd, confirm_col_btn, confirm_col_out]))

with confirm_col_out:
    if SELECTED_COLUMN in plottable:
        print(f"Confirmed column: {SELECTED_COLUMN} (already confirmed - you can move on)")
    else:
        print(f"Not confirmed yet - will default to '{_initial_col}'.")

### Apply the selection and plot

In [ ]:
SELECTED_COLUMN = globals().get("SELECTED_COLUMN")
if SELECTED_COLUMN not in plottable:
    SELECTED_COLUMN = plottable[0]
    print(f"No column confirmed - defaulting to '{SELECTED_COLUMN}'.")
DATA_COLUMN = SELECTED_COLUMN

# The quality-flag column is not identical across products, so it is resolved against the data
# rather than assumed, and the notebook says what it used.
if f"{DATA_COLUMN}-Flag" in df.columns:
    QUALITY_FLAG = f"{DATA_COLUMN}-Flag"
elif DEFAULT_QUALITY_FLAG in df.columns:
    QUALITY_FLAG = DEFAULT_QUALITY_FLAG
else:
    QUALITY_FLAG = None

if QUALITY_FLAG:
    df_quality = df[df[QUALITY_FLAG] == VALUE_ACCEPT_QUALITY].copy()
    if df_quality.empty:
        observed = sorted(str(v) for v in df[QUALITY_FLAG].dropna().unique())[:10]
        print(f"WARNING: no row has {QUALITY_FLAG} == '{VALUE_ACCEPT_QUALITY}'. "
              f"Values present: {observed}. Plotting unfiltered data.")
        df_quality = df.copy()
else:
    print("No quality-flag column found - plotting unfiltered data.")
    df_quality = df.copy()

df_quality[TIME_COLUMN] = pd.to_datetime(df_quality[TIME_COLUMN], errors="coerce")
df_quality = df_quality.dropna(subset=[TIME_COLUMN, DATA_COLUMN])
print(f"{len(df_quality)} of {len(df)} rows kept"
      + (f" after filtering on {QUALITY_FLAG}." if QUALITY_FLAG else "."))

x_vt = column_value_types.get(TIME_COLUMN)
y_vt = column_value_types.get(DATA_COLUMN)
x_axis_label = f"{x_vt.self.label} ({TIME_COLUMN})" if x_vt else TIME_COLUMN
y_axis_label = (f"{y_vt.self.label} [{y_vt.unit}]" if y_vt and getattr(y_vt, "unit", None)
                else (y_vt.self.label if y_vt else DATA_COLUMN))

plot = df_quality.plot(x=TIME_COLUMN, y=DATA_COLUMN, grid=True, title=station_name,
                       style="o", markersize=3)
plot.set(xlabel=x_axis_label, ylabel=y_axis_label)
register_figure(plot.get_figure(), "single_series")
plt.show()

## 7. Combine all data objects of this station and data type

Every object listed in section 4 is fetched and merged on the timestamp. Columns are renamed with
the station code and sampling height, so that series coming from different heights stay distinct.

Unlike the original version, the columns to keep are **not** specified in advance: every numeric
column found is merged, and you choose what to plot afterwards.

In [ ]:
renamed_data_columns = []
y_axis_labels = []
merged_df = pd.DataFrame(columns=[TIME_COLUMN])
n_used = n_skipped = 0

for dobj, arrs in data.batch_get_columns_as_arrays(station_data_objects[:MAX_OBJECTS_TO_COMBINE]):
    d = pd.DataFrame(arrs)

    flag_col = DEFAULT_QUALITY_FLAG if DEFAULT_QUALITY_FLAG in d.columns else None
    if flag_col:
        d = d[d[flag_col] == VALUE_ACCEPT_QUALITY].copy()
    else:
        d = d.copy()

    if TIME_COLUMN not in d.columns:
        print(f"No '{TIME_COLUMN}' column in {dobj.filename} - skipped. "
              f"Available: {list(d.columns)}")
        n_skipped += 1
        continue

    station_id = dobj.station_uri.split("_")[-1]
    suffix = f"_{station_id}_{dobj.sampling_height}" if dobj.sampling_height else f"_{station_id}"

    # Ancillary statistics carry the same generic names in every object ("Stdev", "NbPoints"),
    # so two objects at the same station and height would collide after renaming. They are not
    # measurement series, so they are left out.
    def _is_ancillary(col):
        return col.endswith(("Stdev", "NbPoints", "Flag"))

    value_cols = [c for c in d.columns
                  if c != TIME_COLUMN and not _is_ancillary(c)
                  and pd.api.types.is_numeric_dtype(d[c])]
    if not value_cols:
        n_skipped += 1
        continue

    # Even so, two objects can legitimately share station, height and species; disambiguate
    # rather than letting pandas raise a MergeError on duplicate columns.
    rename = {}
    for c in value_cols:
        new = c + suffix
        n = 2
        while new in merged_df.columns:
            new = f"{c}{suffix}_{n}"
            n += 1
        rename[c] = new
    d = d.rename(columns=rename)

    obj_meta = meta.get_dobj_meta(dobj)
    vtypes = {col.label: col.valueType for col in obj_meta.specificInfo.columns}
    for c in value_cols:
        new = rename[c]
        if new not in renamed_data_columns:
            renamed_data_columns.append(new)
            vt = vtypes.get(c)
            y_axis_labels.append(f"{vt.self.label} [{vt.unit}]" if vt and getattr(vt, "unit", None)
                                 else new)

    merged_df = pd.merge(merged_df, d[[TIME_COLUMN] + list(rename.values())],
                         on=TIME_COLUMN, how="outer")
    n_used += 1

if merged_df.empty or not renamed_data_columns:
    raise RuntimeError("No data could be merged for this station and data type.")

merged_df[TIME_COLUMN] = pd.to_datetime(merged_df[TIME_COLUMN], errors="coerce")
merged_df = merged_df.dropna(subset=[TIME_COLUMN]).sort_values(TIME_COLUMN)

print(f"Merged {n_used} object(s)" + (f", {n_skipped} skipped" if n_skipped else "")
      + f"; {len(renamed_data_columns)} series, {len(merged_df)} rows.")
display(merged_df)

## 8. Plot several series together

Select the series to plot. Species with very different value ranges do not sit well on one axis —
the two-axis plot in the next section is better for that.

In [ ]:
label_by_col = dict(zip(renamed_data_columns, y_axis_labels))

series_select = widgets.SelectMultiple(
    options=[(f"{c} - {label_by_col[c]}", c) for c in renamed_data_columns],
    value=tuple(renamed_data_columns[:3]),
    description="Series:", style={"description_width": "initial"},
    rows=min(10, max(3, len(renamed_data_columns))),
    layout=widgets.Layout(width="620px"))
plot_btn = widgets.Button(description="Plot selected series", button_style="success", icon="check")
plot_out = widgets.Output()

def _plot_series(_=None):
    with plot_out:
        plot_out.clear_output(wait=True)
        cols = list(series_select.value)
        if not cols:
            print("Select at least one series.")
            return
        ax = merged_df.plot(x=TIME_COLUMN, y=cols, grid=True, style="o", markersize=3,
                            figsize=(11, 5), title=station_name)
        ax.legend([f"{c} ({label_by_col[c]})" for c in cols], fontsize=8)
        ax.set_xlabel("Time")
        register_figure(ax.get_figure(), "selected_series")
plt.show()

plot_btn.on_click(_plot_series)
display(widgets.VBox([series_select, plot_btn, plot_out]))
_plot_series()

### Zoom on the latest year

In [ ]:
cols = list(series_select.value) or renamed_data_columns[:3]
latest_year = merged_df[TIME_COLUMN].dt.year.max()
subset = merged_df[merged_df[TIME_COLUMN].dt.year == latest_year]

if subset.empty:
    print("No data in the latest year.")
else:
    ax = subset.plot(x=TIME_COLUMN, y=cols, grid=True, style="o", markersize=3,
                     figsize=(11, 5), title=f"{station_name} - {latest_year}")
    ax.legend([f"{c} ({label_by_col[c]})" for c in cols], fontsize=8)
    ax.set_xlabel("Time")
    register_figure(ax.get_figure(), "latest_year")
plt.show()

## 9. Plot two series on separate axes

Useful when the two variables have different units or value ranges. Both series are chosen from the
dropdowns, so nothing has to be typed — in particular you do not have to reproduce by hand a column
name such as `n2o_HTM_150.0`, which encodes station and sampling height.

In [ ]:
left_dd = widgets.Dropdown(
    options=[(f"{c} - {label_by_col[c]}", c) for c in renamed_data_columns],
    value=renamed_data_columns[0], description="Left axis:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="620px"))
right_dd = widgets.Dropdown(
    options=[("(none)", None)] + [(f"{c} - {label_by_col[c]}", c) for c in renamed_data_columns],
    value=renamed_data_columns[1] if len(renamed_data_columns) > 1 else None,
    description="Right axis:", style={"description_width": "initial"},
    layout=widgets.Layout(width="620px"))
two_btn = widgets.Button(description="Plot", button_style="success", icon="check")
two_out = widgets.Output()

def _plot_two(_=None):
    with two_out:
        two_out.clear_output(wait=True)
        left, right = left_dd.value, right_dd.value
        fig, ax1 = plt.subplots(figsize=(11, 5))
        ax1.plot(merged_df[TIME_COLUMN], merged_df[left], "b.", markersize=3)
        ax1.set_xlabel("Time")
        ax1.set_ylabel(label_by_col[left], color="b")
        ax1.tick_params(axis="y", labelcolor="b")
        ax1.set_title(station_name)

        if right and right != left:
            ax2 = ax1.twinx()
            ax2.plot(merged_df[TIME_COLUMN], merged_df[right], "r.", markersize=3)
            ax2.set_ylabel(label_by_col[right], color="r")
            ax2.tick_params(axis="y", labelcolor="r")
        elif right == left:
            print("The two axes show the same series - pick a different one on the right.")

        ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
        ax1.tick_params(axis="x", rotation=45)
        ax1.grid(True)
        fig.tight_layout()
        register_figure(fig, "two_axes")
        plt.show()

two_btn.on_click(_plot_two)
display(widgets.VBox([left_dd, right_dd, two_btn, two_out]))
_plot_two()

<a id="export"></a>
## Take the results with you

The figures above are only displayed in the notebook: nothing has been written to disk while you
worked, and the output folder is emptied every time the notebook is run again.

The cell below writes out the figures that were drawn, packs them into a single zip and gives you a
download link. Run it before starting a new run if you want to keep anything.

The archive is written into an `exports/` folder next to the notebook. That folder is emptied at the start of each run as well, so download what you need before running the notebook again.

In [ ]:
import os
import zipfile

from IPython.display import HTML

RESULT_SUFFIXES = (".png", ".csv", ".txt")


def _human(n):
    for unit in ("B", "kB", "MB", "GB"):
        if n < 1024 or unit == "GB":
            return f"{n:.0f} {unit}" if unit == "B" else f"{n:.1f} {unit}"
        n /= 1024


bundle_btn = widgets.Button(description="Create archive", button_style="success",
                            icon="file-archive-o")
bundle_out = widgets.Output()


def _make_bundle(_=None):
    with bundle_out:
        bundle_out.clear_output()

        # The figures live in memory only: the inline backend closes them at the end of each
        # cell, so they are written out here rather than while plotting.
        written = []
        for stem, fig in DRAWN_FIGURES.items():
            path = OUTPUT_DIR / f"{stem}.png"
            if not path.exists():
                fig.savefig(path, dpi=150, bbox_inches="tight")
                written.append(path.name)
        if written:
            print(f"Wrote {len(written)} figure(s): {', '.join(written)}")

        files = [f for f in sorted(OUTPUT_DIR.rglob("*"))
                 if f.is_file() and f.suffix.lower() in RESULT_SUFFIXES]
        if not files:
            print("Nothing to pack yet: no figure has been drawn in this session. "
                  "Run the plot cells above first.")
            return

        archive = EXPORT_DIR / f"{NOTEBOOK_SLUG}_results.zip"
        with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
            for f in files:
                zf.write(f, f.relative_to(OUTPUT_DIR))

        total = sum(f.stat().st_size for f in files)
        print(f"{len(files)} file(s), {_human(total)} -> {_human(archive.stat().st_size)} compressed")
        for f in files:
            print(f"   {f.relative_to(OUTPUT_DIR)}")

        # A plain link would make JupyterLab try to *open* the zip, which fails with "is not
        # UTF-8 encoded". The /files/ endpoint hands it to the browser instead, and under
        # JupyterHub it has to carry the /user/<name>/ prefix or it reaches the hub and
        # returns an HTML error page.
        try:
            rel = archive.resolve().relative_to(Path.home())
        except ValueError:
            rel = archive.name
        _prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "/").rstrip("/")
        display(HTML(
            f'<p style="margin-top:8px"><a href="{_prefix}/files/{rel}?download=1" download '
            f'target="_blank" style="padding:8px 14px;background:#1B4053;color:white;'
            f'text-decoration:none;border-radius:5px;display:inline-block">'
            f'Download {archive.name}</a></p>'
            f'<p style="font-size:90%;color:#555">If the link does nothing, right-click '
            f'<code>{archive.name}</code> in the file browser on the left and choose '
            f'<i>Download</i>.</p>'))


bundle_btn.on_click(_make_bundle)
display(widgets.VBox([bundle_btn, bundle_out]))
print("Press 'Create archive' when you want to keep the figures of this run.")